# Homework 3: Classification

Questions: [ML Zoomcamp 2026 – Homework 3](https://github.com/DataTalksClub/machine-learning-zoomcamp/blob/main/cohorts/2026/homework/03-classification/homework.md)

Dataset: [2026 Course Lead Scoring](https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv)

Goal: predict `converted` (whether the client signed up) with logistic regression.

In [1]:
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, mutual_info_score
from sklearn.model_selection import train_test_split

## Data preparation

Check for missing values. Fill categorical features with `'NA'` and numerical features with `0.0`.

In [2]:
categorical = ["lead_source", "industry", "employment_status", "location"]
numerical = ["annual_income", "number_of_courses_viewed", "interaction_count", "lead_score"]
target = "converted"

df = pd.read_csv("data/course_lead_scoring_2026.csv")
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [3]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [4]:
df[categorical] = df[categorical].fillna("NA")
df[numerical] = df[numerical].fillna(0.0)
df.isnull().sum().sum()

np.int64(0)

## Q1. Mode of `industry`

What is the most frequent observation (mode) for the column `industry`?

- `NA`
- `technology`
- `healthcare`
- `retail`

In [5]:
df["industry"].value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

**Answer:** `technology`

## Q2. Correlation matrix

Which two features have the biggest correlation? Only consider these pairs:

- `interaction_count` and `lead_score`
- `number_of_courses_viewed` and `lead_score`
- `number_of_courses_viewed` and `interaction_count`
- `annual_income` and `interaction_count`

In [6]:
corr = df[numerical].corr()
corr.round(3)

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000,0.161,0.123,0.229
number_of_courses_viewed,0.161,1.000,0.722,0.757
interaction_count,0.123,0.722,1.000,0.916
lead_score,0.229,0.757,0.916,1.000


In [7]:
pairs = [
    ("interaction_count", "lead_score"),
    ("number_of_courses_viewed", "lead_score"),
    ("number_of_courses_viewed", "interaction_count"),
    ("annual_income", "interaction_count"),
]
for pair in pairs:
    print(f"{' / '.join(pair)}: {corr.loc[pair]:.4f}")

interaction_count / lead_score: 0.9157
number_of_courses_viewed / lead_score: 0.7572
number_of_courses_viewed / interaction_count: 0.7216
annual_income / interaction_count: 0.1228


**Answer:** `interaction_count` and `lead_score` (0.916)

## Split the data

Use the exact `train_test_split` calls from the homework (60/20/20) and remove `converted` from the feature frames.

In [8]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

y_train = df_train.pop(target).values
y_val = df_val.pop(target).values
y_test = df_test.pop(target).values

len(df_train), len(df_val), len(df_test)

(3000, 1000, 1000)

## Q3. Mutual information

Mutual information between `converted` and each categorical variable on the training set, rounded to 2 decimals.

- `industry`
- `location`
- `lead_source`
- `employment_status`

In [9]:
mi = df_train[categorical].apply(lambda col: round(mutual_info_score(y_train, col), 2))
mi.sort_values(ascending=False)

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

**Answer:** `lead_source` (0.03)

## Q4. Logistic regression

One-hot encode the categorical variables, fit `LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)` on train, and report validation accuracy rounded to 2 decimals.

- 0.55
- 0.65
- 0.75
- 0.85

In [10]:
def train_and_score(features, C=1.0):
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(df_train[features].to_dict(orient="records"))
    X_val = dv.transform(df_val[features].to_dict(orient="records"))

    model = LogisticRegression(solver="liblinear", C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    return accuracy_score(y_val, model.predict(X_val))

In [11]:
features = categorical + numerical
base = train_and_score(features)
round(base, 2)

0.65

**Answer:** 0.65

## Q5. Feature elimination

Drop each feature in turn, retrain, and compute `original accuracy - accuracy without feature`. Which of these has the smallest difference?

- `'lead_source'`
- `'number_of_courses_viewed'`
- `'interaction_count'`

In [12]:
diffs = {}
for feature in features:
    accuracy = train_and_score([f for f in features if f != feature])
    diffs[feature] = base - accuracy
    print(f"without {feature}: {accuracy:.4f} (diff {diffs[feature]:+.4f})")

without lead_source: 0.6420 (diff +0.0030)
without industry: 0.6450 (diff +0.0000)
without employment_status: 0.6450 (diff +0.0000)
without location: 0.6450 (diff +0.0000)
without annual_income: 0.7240 (diff -0.0790)
without number_of_courses_viewed: 0.6430 (diff +0.0020)


without interaction_count: 0.6010 (diff +0.0440)
without lead_score: 0.6440 (diff +0.0010)


**Answer:** `number_of_courses_viewed` (+0.002, vs +0.003 for `lead_source` and +0.044 for `interaction_count`)

Dropping `annual_income` actually *improves* accuracy by ~8 points: it's unscaled (tens of thousands) so it dominates the regularized model.

## Q6. Regularization

Try `C` in `[0.000001, 0.00001, 0.0001, 0.001]` and round validation accuracy to 3 decimals. If there's a tie, pick the smallest `C`.

- 0.000001
- 0.00001
- 0.0001
- 0.001

In [13]:
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    print(f"C={C}: {round(train_and_score(features, C=C), 3)}")

C=1e-06: 0.598
C=1e-05: 0.598
C=0.0001: 0.613


C=0.001: 0.645


**Answer:** 0.001 (0.645)

## Summary

| Question | Answer |
|---|---|
| Q1. Mode of `industry` | `technology` |
| Q2. Most correlated pair | `interaction_count` and `lead_score` |
| Q3. Biggest mutual information | `lead_source` |
| Q4. Validation accuracy | 0.65 |
| Q5. Smallest difference | `number_of_courses_viewed` |
| Q6. Best `C` | 0.001 |